# Tarea 2 - Selección, limpieza y alistamiento de datos

## Pregunta de negocio 2 — Dimensión territorial

**Rol:** Ingeniería de datos / Análisis de datos
**Integrante:** Cristian Camilo Rodríguez Cagueñas

**Pregunta inicialmente planteada:**

> ¿Cómo se distribuyen los recursos contratados de INVIAS en el territorio y qué
> departamentos o ciudades concentran la mayor contratación, considerando además
> su evolución en el tiempo?

**Objetivo de este notebook**

Auditar la disponibilidad real de información territorial en el conjunto de datos
de contratación de INVIAS extraído de SECOP II, con el fin de determinar si la
pregunta de negocio planteada puede ser respondida con las variables disponibles.

La auditoría se realiza antes de cualquier proceso de limpieza o transformación,
de manera que las decisiones metodológicas posteriores queden sustentadas en
evidencia sobre la fuente.

## 1. Auditoría inicial de la dimensión territorial

Se verifica la estructura general del conjunto de datos y, en particular, el
contenido de las variables que en principio permitirían ubicar geográficamente
la contratación de la entidad.

In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import re
import unicodedata

In [2]:
# Cargamos el archivo CSV en un DataFrame de pandas
# El archivo se encuentra en la ruta relativa "../../invias.csv"
df = pd.read_csv("../../invias.csv", low_memory=False)

print("Número de filas:", df.shape[0])
print("Número de columnas:", df.shape[1])

Número de filas: 25605
Número de columnas: 89


### 1.1 Identificación de las variables territoriales candidatas

A partir de la pregunta de negocio se identifican las variables del conjunto de
datos que podrían aportar información sobre la ubicación geográfica de la
contratación.

In [3]:
# Definimos las variables candidatas para el análisis territorial
variables_territoriales = [
    "departamento",
    "ciudad",
    "localizacion",
    "direccion_de_ejecucion_del_contrato",
    "orden",
]

df[variables_territoriales].head(10)

,departamento,ciudad,localizacion,direccion_de_ejecucion_del_contrato,orden
0,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
1,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
2,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
3,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
4,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
5,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
6,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
7,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
8,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional
9,Distrito Capital de Bogotá,Bogotá,"Colombia, Bogotá, Bogotá",NaN,Nacional


### 1.2 Evaluación de cardinalidad y completitud

Para cada variable candidata se calcula el número de valores únicos y el número
de valores faltantes. Una variable con un único valor no aporta capacidad
diferenciadora, y una variable mayoritariamente vacía no permite sustentar un
análisis.

In [4]:
# Calculamos cardinalidad y completitud de las variables territoriales
auditoria_territorial = pd.DataFrame({
    "valores_unicos": df[variables_territoriales].nunique(dropna=True),
    "no_nulos": df[variables_territoriales].notna().sum(),
    "faltantes": df[variables_territoriales].isna().sum(),
})
auditoria_territorial["%_faltantes"] = (
    100 * auditoria_territorial["faltantes"] / len(df)
).round(2)

auditoria_territorial

,valores_unicos,no_nulos,faltantes,%_faltantes
departamento,1,25605,0,0.00
ciudad,1,25605,0,0.00
localizacion,1,25605,0,0.00
direccion_de_ejecucion_del_contrato,1,3,25602,99.99
orden,1,25605,0,0.00


### 1.3 Inspección del contenido de cada variable

Se revisa la distribución de valores de cada variable candidata para determinar
qué información contienen realmente.

In [5]:
# Revisamos la distribución de valores de cada variable territorial
for columna in variables_territoriales:
    print(f"--- {columna} ---")
    print(df[columna].value_counts(dropna=False).head(5))
    print()

--- departamento ---
departamento
Distrito Capital de Bogotá    25605
Name: count, dtype: int64

--- ciudad ---
ciudad
Bogotá    25605
Name: count, dtype: int64

--- localizacion ---
localizacion
Colombia, Bogotá,  Bogotá    25605
Name: count, dtype: int64

--- direccion_de_ejecucion_del_contrato ---
direccion_de_ejecucion_del_contrato
NaN            25602
No definido        3
Name: count, dtype: int64

--- orden ---
orden
Nacional    25605
Name: count, dtype: int64



### 1.4 Hallazgo principal de la auditoría

La revisión evidencia que **ninguna de las variables geográficas estructuradas
contiene información sobre el lugar de ejecución de los contratos**:

- `departamento` registra el valor *Distrito Capital de Bogotá* en el 100 % de los registros.
- `ciudad` registra el valor *Bogotá* en el 100 % de los registros.
- `localizacion` registra el valor *Colombia, Bogotá, Bogotá* en el 100 % de los registros.
- `direccion_de_ejecucion_del_contrato` se encuentra vacía en prácticamente la totalidad
  del conjunto de datos.
- `orden` registra el valor *Nacional* en el 100 % de los registros, consistente con la
  naturaleza de la entidad.

Estas variables no describen el lugar donde se ejecuta el contrato, sino el
**domicilio de la entidad contratante**. Dado que INVIAS es una entidad del orden
nacional con sede en Bogotá, el registro es correcto desde el punto de vista del
SECOP II, pero carece de valor analítico para una pregunta territorial.

Se verifica a continuación de manera explícita esta afirmación.

In [6]:
# Verificamos explícitamente que las variables geográficas son constantes
for columna in ["departamento", "ciudad", "localizacion", "orden"]:
    unicos = df[columna].unique()
    print(f"{columna}: {len(unicos)} valor(es) único(s) -> {unicos}")

print()
print("direccion_de_ejecucion_del_contrato:")
print("  registros no nulos:", df["direccion_de_ejecucion_del_contrato"].notna().sum())
print("  valores presentes:", df["direccion_de_ejecucion_del_contrato"].dropna().unique())

departamento: 1 valor(es) único(s) -> <StringArray>
['Distrito Capital de Bogotá']
Length: 1, dtype: str
ciudad: 1 valor(es) único(s) -> <StringArray>
['Bogotá']
Length: 1, dtype: str
localizacion: 1 valor(es) único(s) -> <StringArray>
['Colombia, Bogotá,  Bogotá']
Length: 1, dtype: str
orden: 1 valor(es) único(s) -> <StringArray>
['Nacional']
Length: 1, dtype: str

direccion_de_ejecucion_del_contrato:
  registros no nulos: 3
  valores presentes: <StringArray>
['No definido']
Length: 1, dtype: str


### 1.5 Consecuencia sobre la pregunta de negocio

La pregunta de negocio planteada inicialmente presupone la existencia de una
variable que identifique el departamento o municipio de ejecución del contrato.
La auditoría demuestra que **dicha variable no existe en la fuente**.

En consecuencia, la pregunta no puede responderse a partir de los campos
estructurados del conjunto de datos. Antes de descartar la dimensión territorial
del proyecto, se evalúa si la información existe en otras variables del conjunto
de datos bajo un formato no estructurado.

## 2. Búsqueda de fuentes alternativas de información territorial

Se inspeccionan las variables de texto libre del conjunto de datos, en las cuales
la entidad describe el alcance de cada contrato.

In [7]:
# Revisamos la completitud de las variables de texto libre
variables_texto = ["objeto_del_contrato", "descripcion_del_proceso"]

pd.DataFrame({
    "no_nulos": df[variables_texto].notna().sum(),
    "faltantes": df[variables_texto].isna().sum(),
    "%_faltantes": (100 * df[variables_texto].isna().sum() / len(df)).round(2),
})

,no_nulos,faltantes,%_faltantes
objeto_del_contrato,20699,4906,19.16
descripcion_del_proceso,20718,4887,19.09


In [8]:
# Inspeccionamos una muestra del objeto contractual
muestra = df["objeto_del_contrato"].dropna().sample(10, random_state=42)

for texto in muestra:
    print("-", texto[:220].replace("\n", " "))
    print()

- INTERVENTORÍA TÉCNICA; ADMINISTRATIVA; FINANCIERA Y AMBIENTAL PARA EL MEJORAMIENTO DE LAS VÍAS TERCIARIAS EN EL MARCO DE LA IMPLEMENTACION DE LOS ACUERDOS DE PAZ EN EL MUNICIPIO  DE GUATICA; RISARALDA.

- PRESTAR SERVICIOS PROFESIONALES EN TEMAS ADMINISTRATIVOS Y FINANCIEROS PARA DESARROLLAR Y ACOMPAÑAR DESDE LA SUBDIRECCIÓN DE ESTUDIOS E INNOVACIÓN LOS PROYECTOS DE COMPETENCIA DE ÉSTA Y LAS DEMÁS UNIDADES EJECUTORAS DEL 

- AUNAR ESFUERZOS ENTRE EL INSTITUTO NACIONAL DE VÍAS Y LA JUNTA DE ACCIÓN COMUNAL DE LA VEREDA LIMONES; DEL MUNICIPIO DE ZARZAL; DEPARTAMENTO DEL VALLE DEL CAUCA; PARA EL MEJORAMIENTO VIAL EN MARCO DEL PROGRAMA CAMINOS CO

- AUNAR ESFUERZOS ENTRE EL INSTITUTO NACIONAL DE VIAS Y LA JUNTA DE ACCIÓN COMUNAL DE LA VEREDA AGUA FRIA DEL MUNICIPIO DE HOBO; DEL DEPARTAMENTO DE HUILA PARA EL MEJORAMIENTO VIAL EN EL MARCO DEL PROGRAMA CAMINOS COMUNITA

- AUNAR ESFUERZOS ENTRE EL INSTITUTO NACIONAL DE VIAS Y LA JUNTA DE ACCIÓN COMUNAL DE LA VEREDA PEÑA BLANCA DEL MUNICIPIO D

### 2.1 Patrones territoriales identificados en el texto libre

La inspección de las variables `objeto_del_contrato` y `descripcion_del_proceso`
evidencia que la información territorial **sí está presente**, expresada mediante
al menos cuatro patrones:

1. **Mención explícita del departamento**
   *"...DEL MUNICIPIO DE MAJAGUAL; DEPARTAMENTO DE SUCRE..."*

2. **Mención del municipio acompañada del departamento**
   *"...GIRALDO - CUAJARON; ... DEL MUNICIPIO DE GIRALDO - ANTIOQUIA."*

3. **Dirección Territorial del INVIAS**, que corresponde a la unidad administrativa
   de la entidad encargada de gestionar la vía
   *"...Dirección Territorial Santander; Vía Código: 45A08 Bucaramanga - San Alberto..."*

4. **Código de vía y abscisado (PR)**, que identifica el corredor vial
   *"...Vía Código: 45A08 ...; PR18+0000 - PR 60+0000"*

Adicionalmente se observa un conjunto amplio de contratos cuyo objeto corresponde
a **prestación de servicios profesionales de apoyo a la gestión**, los cuales por
su naturaleza no están asociados a un territorio específico sino a la operación
central de la entidad.

Esta observación es relevante: la ausencia de información territorial en una
parte del conjunto de datos no necesariamente constituye un problema de calidad,
sino que puede reflejar la composición real de la contratación de la entidad.

## 3. Prueba de concepto: cobertura potencial de la extracción territorial

Antes de decidir si la dimensión territorial puede sostener una pregunta de
negocio, se estima qué proporción del conjunto de datos podría ser
territorializada a partir del texto libre.

Para esta prueba se emplea un diccionario de los 32 departamentos de Colombia y
Bogotá D.C., aplicado sobre el texto normalizado. Se trata de una estimación
preliminar; la construcción definitiva de la variable se desarrolla en el
notebook `02_construccion_territorial_p2.ipynb`.

In [9]:
# Definimos una función de normalización de texto
# Elimina tildes, convierte a mayúsculas y unifica espacios y signos de puntuación
def normalizar_texto(valor):
    texto = unicodedata.normalize("NFKD", str(valor))
    texto = texto.encode("ascii", "ignore").decode()
    texto = texto.upper()
    texto = re.sub(r"[^A-Z0-9 ]", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


# Concatenamos las dos variables de texto que describen el alcance del contrato
texto_contrato = (
    df["objeto_del_contrato"].fillna("") + " " + df["descripcion_del_proceso"].fillna("")
).map(normalizar_texto)

print("Registros con algún texto disponible:", (texto_contrato.str.len() > 0).sum())

Registros con algún texto disponible: 20717


In [10]:
# Diccionario preliminar de departamentos (prueba de concepto)
departamentos_poc = [
    "AMAZONAS", "ANTIOQUIA", "ARAUCA", "ATLANTICO", "BOLIVAR", "BOYACA",
    "CALDAS", "CAQUETA", "CASANARE", "CAUCA", "CESAR", "CHOCO", "CORDOBA",
    "CUNDINAMARCA", "GUAINIA", "GUAVIARE", "HUILA", "LA GUAJIRA", "MAGDALENA",
    "META", "NARINO", "NORTE DE SANTANDER", "PUTUMAYO", "QUINDIO", "RISARALDA",
    "SANTANDER", "SUCRE", "TOLIMA", "VALLE DEL CAUCA", "VAUPES", "VICHADA",
    "SAN ANDRES", "BOGOTA",
]

# Contamos cuántos departamentos distintos menciona cada contrato
patrones_poc = {
    d: re.compile(r"\b" + d.replace(" ", r"\s+") + r"\b") for d in departamentos_poc
}

df["n_departamentos_poc"] = texto_contrato.map(
    lambda t: sum(1 for p in patrones_poc.values() if p.search(t))
)

df["n_departamentos_poc"].value_counts().sort_index().head(6)

n_departamentos_poc
0    16978
1     7534
2     1003
3       66
4       19
5        2
Name: count, dtype: int64

### 3.1 Cobertura en número de contratos

In [11]:
# Clasificamos los contratos según la información territorial detectada
df["grupo_territorial_poc"] = np.select(
    [df["n_departamentos_poc"] == 0, df["n_departamentos_poc"] == 1],
    ["Sin mención territorial", "Un departamento"],
    default="Varios departamentos",
)

cobertura_contratos = df["grupo_territorial_poc"].value_counts().to_frame("contratos")
cobertura_contratos["%"] = (100 * cobertura_contratos["contratos"] / len(df)).round(1)

cobertura_contratos

,contratos,%
grupo_territorial_poc,,
Sin mención territorial,16978,66.3
Un departamento,7534,29.4
Varios departamentos,1093,4.3


### 3.2 Cobertura en valor contratado

El indicador anterior mide cobertura en número de contratos. Sin embargo, para
una pregunta sobre **distribución de recursos**, el criterio determinante es qué
proporción del **valor contratado** resulta territorializable.

Para este cálculo se excluyen de manera preliminar:

- los contratos sin valor registrado o con valor igual a cero,
- los contratos con valores superiores a un billón de pesos, identificados durante
  la auditoría de la Pregunta 3 como registros que requieren validación de calidad
  de datos antes de ser incorporados a agregaciones.

Ambas exclusiones se documentan y no modifican el conjunto de datos original.

In [12]:
# Construimos el subconjunto con valor contractual válido para el cálculo de cobertura
UMBRAL_VALOR_ATIPICO = 1e12  # 1 billón de pesos

df_valor = df[
    df["valor_del_contrato"].notna()
    & (df["valor_del_contrato"] > 0)
    & (df["valor_del_contrato"] < UMBRAL_VALOR_ATIPICO)
].copy()

print("Contratos con valor contractual válido:", len(df_valor))
print("Contratos excluidos por valor atípico (> 1 billón):",
      (df["valor_del_contrato"] >= UMBRAL_VALOR_ATIPICO).sum())

Contratos con valor contractual válido: 19617
Contratos excluidos por valor atípico (> 1 billón): 15


In [13]:
# Calculamos la cobertura territorial en número de contratos y en valor
cobertura_valor = df_valor.groupby("grupo_territorial_poc").agg(
    contratos=("id_contrato", "size"),
    valor_total=("valor_del_contrato", "sum"),
)

cobertura_valor["%_contratos"] = (
    100 * cobertura_valor["contratos"] / cobertura_valor["contratos"].sum()
).round(1)
cobertura_valor["valor_billones"] = (cobertura_valor["valor_total"] / 1e12).round(2)
cobertura_valor["%_valor"] = (
    100 * cobertura_valor["valor_total"] / cobertura_valor["valor_total"].sum()
).round(1)

cobertura_valor[["contratos", "%_contratos", "valor_billones", "%_valor"]]

,contratos,%_contratos,valor_billones,%_valor
grupo_territorial_poc,,,,
Sin mención territorial,11181,57.0,8.65,22.1
Un departamento,7361,37.5,18.06,46.1
Varios departamentos,1075,5.5,12.43,31.8


### 3.3 Caracterización de los contratos sin mención territorial

Se verifica la hipótesis planteada en la sección 2.1: los contratos sin
información territorial corresponderían principalmente a contratación de apoyo a
la operación central de la entidad y no a fallas de registro.

In [14]:
# Revisamos qué tipos de contrato predominan entre los registros sin mención territorial
sin_territorio = df_valor[df_valor["grupo_territorial_poc"] == "Sin mención territorial"]

composicion = (
    100 * sin_territorio["tipo_de_contrato"].value_counts(normalize=True)
).round(1).to_frame("%_contratos")

composicion.head(8)

,%_contratos
tipo_de_contrato,
Prestación de servicios,84.1
Obra,7.2
Suministros,3.4
Interventoría,2.4
Otro,1.0
Consultoría,0.9
Compraventa,0.7
Seguros,0.3


### 3.4 Cobertura temporal

Dado que la pregunta de negocio incluye la evolución en el tiempo, se verifica el
rango y la densidad temporal de la variable `fecha_de_firma`.

In [15]:
# Convertimos la fecha de firma y revisamos su distribución anual
fecha_firma = pd.to_datetime(df_valor["fecha_de_firma"], errors="coerce", utc=True)

print("Rango temporal:", fecha_firma.min().date(), "->", fecha_firma.max().date())
print()
print(fecha_firma.dt.year.value_counts().sort_index().to_string())

Rango temporal: 2017-11-29 -> 2026-08-06

fecha_de_firma
2017.0      47
2018.0    1070
2019.0    1253
2020.0    2037
2021.0    2019
2022.0    1504
2023.0    3871
2024.0    2952
2025.0    1613
2026.0     877


## 4. Reformulación de la pregunta de negocio

### 4.1 Sustento de la reformulación

La auditoría permite establecer tres hechos:

1. **Las variables geográficas estructuradas no contienen el lugar de ejecución.**
   Registran el domicilio de la entidad y son constantes en todo el conjunto de datos.

2. **La información territorial existe en el texto libre** y es recuperable de forma
   sistemática, aunque con cobertura parcial.

3. **La cobertura parcial no es aleatoria.** Los contratos sin mención territorial
   corresponden mayoritariamente a prestación de servicios de apoyo a la operación
   central, y concentran una fracción minoritaria del valor contratado.

Este último punto es determinante: aunque la extracción no cubre la totalidad de
los registros, sí cubre la mayor parte de los **recursos**, que es la unidad de
análisis relevante para una pregunta sobre distribución territorial de la
contratación.

### 4.2 Pregunta reformulada

> **¿Dónde ejecuta INVIAS sus recursos?** Dado que el SECOP II no registra el
> lugar de ejecución del contrato, ¿qué proporción de la contratación de la
> entidad es territorialmente identificable a partir del objeto contractual, y
> cómo se distribuyen y evolucionan en el territorio los recursos que sí lo son?

### 4.3 Preguntas específicas derivadas

1. ¿Qué proporción de los contratos y del valor contratado es territorialmente
   identificable, y cómo ha evolucionado esa trazabilidad en el tiempo?
2. ¿Qué departamentos concentran el mayor valor contratado entre los contratos
   territorialmente identificables?
3. ¿Cómo se comporta esa distribución territorial a lo largo del tiempo y según el
   tipo de contrato y la modalidad de contratación?
4. ¿Qué diferencia existe entre la contratación territorializable (misional) y la
   contratación asociada a la operación central de la entidad?

### 4.4 Valor para el usuario final

El usuario definido para el producto de analítica corresponde a la alta Dirección
del INVIAS y a los organismos de control interno. Para este usuario, el resultado
de la reformulación aporta dos elementos:

- La **distribución territorial de los recursos** entre los contratos identificables.
- Una **medida de trazabilidad territorial** de la contratación de la entidad, es
  decir, qué proporción de los recursos ejecutados no puede ubicarse geográficamente
  a partir de la información publicada en el SECOP II. Este segundo elemento
  constituye en sí mismo un hallazgo de interés para el seguimiento y control.

### 4.5 Alcance y limitaciones declaradas

- La variable territorial es **inferida**, no reportada por la entidad. Se documenta
  como tal en todos los resultados.
- La inferencia se basa en la mención textual de un departamento en el objeto
  contractual, lo cual **no garantiza** que la totalidad de la ejecución ocurra en
  ese departamento.
- Los contratos que mencionan varios departamentos se tratan como una categoría
  propia y **no se replican** entre departamentos, con el fin de no duplicar valor
  en las agregaciones.
- La precisión de la extracción se estima mediante revisión manual de una muestra
  aleatoria, procedimiento que se documenta en el notebook 02.

## 5. Conclusiones de la auditoría

1. El conjunto de datos de INVIAS **no contiene información estructurada sobre el
   lugar de ejecución** de los contratos. Las variables `departamento`, `ciudad`,
   `localizacion` y `orden` son constantes, y `direccion_de_ejecucion_del_contrato`
   se encuentra vacía en prácticamente la totalidad de los registros.

2. En consecuencia, las variables `ciudad`, `localizacion` y `orden` serán eliminadas
   del conjunto de trabajo por no aportar capacidad diferenciadora, en línea con el
   criterio ya aplicado en la Pregunta 3 para `nombre_entidad` y `nit_entidad`.

3. La información territorial **es recuperable a partir del texto libre** del objeto
   contractual y de la descripción del proceso, mediante la construcción de una
   variable derivada.

4. La cobertura estimada de esta extracción alcanza una fracción minoritaria de los
   **contratos** pero mayoritaria del **valor contratado**, y los registros no
   cubiertos corresponden mayoritariamente a prestación de servicios de apoyo a la
   operación central.

5. La pregunta de negocio se reformula para incorporar la trazabilidad territorial
   como parte del objeto de análisis, en lugar de asumir una variable geográfica que
   la fuente no proporciona.

**Siguiente paso:** construcción de la variable territorial y de la base analítica
de la Pregunta 2, desarrollada en `02_construccion_territorial_p2.ipynb`.